# NLP テキスト分類コンペ実践

TF-IDF ベースラインから BERT ファインチューニング、アンサンブルまで NLP コンペの流れを学びます。

> **Kaggle Tutorial** 対応ノートブック（`/competitions/nlp-text-classification`）です。上から順にセルを実行（`Shift + Enter` または「ランタイム → すべてのセルを実行」）して動作を確認できます。

In [ ]:
# [Colab / ローカル環境用セットアップ]
# transformers の確認と、重複のない訓練・テスト分離済みテキスト分類サンプルデータを準備します。
import os
import subprocess
import sys

try:
    import transformers  # noqa: F401
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "transformers"], check=True)

import pandas as pd

def _ensure_nlp_comp_data():
    data_dir = os.path.abspath("./.kaggle_input/nlp-text-classification")
    os.makedirs(data_dir, exist_ok=True)
    orig_read_csv = pd.read_csv
    if not getattr(orig_read_csv, "_is_nlp_comp_wrapped", False):
        def _patched_read_csv(filepath_or_buffer, *args, **kwargs):
            if isinstance(filepath_or_buffer, str) and filepath_or_buffer.startswith("/kaggle/input/<competition>/"):
                if not os.path.exists(filepath_or_buffer):
                    rel = filepath_or_buffer[len("/kaggle/input/<competition>/"):]
                    filepath_or_buffer = os.path.join(data_dir, rel)
            return orig_read_csv(filepath_or_buffer, *args, **kwargs)
        _patched_read_csv._is_nlp_comp_wrapped = True
        pd.read_csv = _patched_read_csv

    train_path = os.path.join(data_dir, "train.csv")
    test_path = os.path.join(data_dir, "test.csv")
    if os.path.exists(train_path) and os.path.exists(test_path):
        print(f"Using existing NLP competition dataset at: {data_dir}")
        return

    pos_subjects = [
        "Stratified cross validation",
        "Out-of-fold feature encoding",
        "Ensembling diverse architectures",
        "Careful pipeline isolation",
        "Domain-driven feature engineering",
        "Calibrated probability blending",
    ]
    pos_outcomes = [
        "kept local validation aligned with the private leaderboard.",
        "improved generalization on unseen test records.",
        "reduced prediction variance across all folds.",
        "boosted downstream classification accuracy reliably.",
    ]
    neg_subjects = [
        "Fitting the vectorizer before splitting",
        "Global missing-value imputation across folds",
        "Overfitting hyperparameters to public scores",
        "Unchecked target leakage in preprocessing",
        "Ignoring severe class imbalance during training",
        "Mixing validation statistics into training features",
    ]
    neg_outcomes = [
        "leaked validation information and distorted evaluation.",
        "caused a severe private leaderboard shakeup.",
        "degraded generalization on held-out test samples.",
        "inflated local metrics while hurting real accuracy.",
    ]

    pos_samples = [f"{s} {o}" for s in pos_subjects for o in pos_outcomes]
    neg_samples = [f"{s} {o}" for s in neg_subjects for o in neg_outcomes]

    train_df = pd.DataFrame({
        "text": pos_samples + neg_samples,
        "label": [1] * len(pos_samples) + [0] * len(neg_samples),
    })
    test_df = pd.DataFrame({
        "text": [
            "Out-of-fold validation prevented leakage and improved model blending.",
            "Target leakage inflated validation metrics and ruined test performance.",
            "Transfer learning with pretrained transformers boosted text accuracy.",
            "Global scaling before splitting leaked statistics and caused shakeup.",
        ]
    })
    train_df.to_csv(train_path, index=False)
    test_df.to_csv(test_path, index=False)
    print(f"NLP competition dataset ready at: {data_dir}")

_ensure_nlp_comp_data()

## 概要

テキスト分類コンペでは、**軽い古典手法で強いベースライン**を作ってから、
**Transformer/BERT のファインチューニング**へ進むのが定石です。L7（NLP）の実践編です。

## 1. EDA とベースライン（TF-IDF + 線形）

In [ ]:
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression

df = pd.read_csv("/kaggle/input/<competition>/train.csv")  # text, label
tfidf = TfidfVectorizer(max_features=50000, ngram_range=(1, 2))
X = tfidf.fit_transform(df["text"])          # 訓練だけで fit
model = LogisticRegression(max_iter=1000).fit(X, df["label"])

まず TF-IDF + ロジスティック回帰で速い基準を作り、交差検証で評価します。

> **Warning:** `TfidfVectorizer` の `fit` は**訓練データだけ**に行い、検証/テストへは `transform`。 全体で fit すると語彙・IDF に検証側が混ざりデータリークになります。

## 2. BERT のファインチューニング（Hugging Face）

In [ ]:
# テストデータを読み込み、訓練データのみで fit した TF-IDF + 線形モデルでテスト予測確率を算出します
test_df = pd.read_csv("/kaggle/input/<competition>/test.csv")
pred_tfidf = model.predict_proba(tfidf.transform(test_df["text"]))[:, 1]

# 続く BERT ファインチューニング用にクラス数と訓練文章リストを設定します
n_classes = int(df["label"].nunique())
texts = df["text"].tolist()

In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification

name = "bert-base-uncased"
tok = AutoTokenizer.from_pretrained(name)
model = AutoModelForSequenceClassification.from_pretrained(name, num_labels=n_classes)
enc = tok(texts, padding=True, truncation=True, max_length=256, return_tensors="pt")
# 小さい学習率(例 2e-5)・短いエポック・検証で早期終了

事前学習済み BERT を各タスクに転移学習します（L7-07）。GPU メモリに応じて
`max_length` とバッチサイズを調整します。

## 3. アンサンブル

In [ ]:
# 訓練ラベルを使って BERT 分類層を含むモデルをファインチューニングし、検証ロス最小の重みでテスト文章を推論します
import numpy as np
import torch
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader, TensorDataset

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.manual_seed(42)
model = model.to(device)

labels_np = df["label"].to_numpy(dtype=np.int64)
tr_idx, va_idx = train_test_split(
    np.arange(len(labels_np)), test_size=0.25, stratify=labels_np, random_state=42
)

tr_dataset = TensorDataset(
    enc["input_ids"][tr_idx],
    enc["attention_mask"][tr_idx],
    torch.tensor(labels_np[tr_idx], dtype=torch.long),
)
tr_loader = DataLoader(tr_dataset, batch_size=8, shuffle=True)
optimizer = torch.optim.AdamW(model.parameters(), lr=2e-5, weight_decay=0.01)

best_val_loss = float("inf")
best_bert_state = None
for _epoch in range(4):
    model.train()
    for b_ids, b_mask, b_y in tr_loader:
        b_ids, b_mask, b_y = b_ids.to(device), b_mask.to(device), b_y.to(device)
        optimizer.zero_grad()
        out = model(input_ids=b_ids, attention_mask=b_mask, labels=b_y)
        out.loss.backward()
        optimizer.step()

    model.eval()
    with torch.no_grad():
        val_out = model(
            input_ids=enc["input_ids"][va_idx].to(device),
            attention_mask=enc["attention_mask"][va_idx].to(device),
            labels=torch.tensor(labels_np[va_idx], dtype=torch.long, device=device),
        )
        val_loss = val_out.loss.item()
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        best_bert_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}

model.load_state_dict(best_bert_state)
model.eval()
test_enc = tok(
    test_df["text"].tolist(), padding=True, truncation=True, max_length=256, return_tensors="pt"
)
with torch.no_grad():
    test_logits = model(
        input_ids=test_enc["input_ids"].to(device),
        attention_mask=test_enc["attention_mask"].to(device),
    ).logits
    pred_bert = torch.softmax(test_logits, dim=-1)[:, 1].cpu().numpy()

In [ ]:
# 誤りの相関が低い TF-IDF線形 と BERT を加重平均
pred = 0.3 * pred_tfidf + 0.7 * pred_bert

In [ ]:
print(f"Fine-tuned BERT best validation loss: {best_val_loss:.4f}")
pd.DataFrame({"text": test_df["text"], "pred_tfidf": pred_tfidf.round(3), "pred_bert": pred_bert.round(3), "blend": pred.round(3)})

性質の異なる古典手法と BERT を ブレンディング すると、多様性が出て安定します（L6）。

> **Note:** 事前学習モデル・外部データには**ライセンスとコンペ規約**があります。利用可否を必ず確認しましょう （L6 / L8 の倫理・規約も参照）。

## 関連トラック

- L7-06 NLP I：前処理〜古典手法
- L7-07 NLP II：Transformer/BERT/Hugging Face
- L6 アンサンブル & コンペワークフロー